# AMP Finder AI — Frozen ESM-2 Embeddings

## Goal

Encode each peptide with the small frozen ESM-2 checkpoint, fit Logistic Regression on those embeddings, and compare it fairly with the feature model using identical held-out rows.

This notebook downloads official model weights on first execution. In Colab, enable a GPU when available.

## Setup

Install optional dependencies before running:

```python
%pip install -q -r requirements-esm.txt
%pip install -q -e .
```

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src/amp_finder").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the AMP Finder AI project.")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
(PROJECT_ROOT / "outputs/figures").mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

In [ ]:
import importlib.util
missing = [name for name in ["torch", "transformers"] if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Install requirements-esm.txt first. Missing: {missing}")

import pandas as pd
import matplotlib.pyplot as plt

from amp_finder.esm import DEFAULT_ESM_MODEL, embed_sequences, save_embedding_bundle
from amp_finder.modeling import load_artifact, save_artifact, train_esm_logistic_regression

real_dataset_path = PROJECT_ROOT / "data/processed/amp_dataset.csv"
demo_dataset_path = PROJECT_ROOT / "data/demo/demo_sequences.csv"
dataset_path = real_dataset_path if real_dataset_path.exists() else demo_dataset_path
dataset = pd.read_csv(dataset_path)
is_demo = dataset_path == demo_dataset_path
print("Dataset:", dataset_path)
print("Status:", "SYNTHETIC UI DEMO — NOT SCIENTIFIC EVIDENCE" if is_demo else "Prepared APD/UniProt demonstration data")

## Steps

### 1. Extract frozen mean-pooled ESM-2 embeddings

In [ ]:
embedding_path = PROJECT_ROOT / ("data/processed/esm2_demo_embeddings.npz" if is_demo else "data/processed/esm2_embeddings.npz")
embeddings = embed_sequences(
    dataset["sequence"],
    model_name=DEFAULT_ESM_MODEL,
    batch_size=16,
    device="auto",
)
save_embedding_bundle(
    embedding_path,
    embeddings=embeddings,
    sequences=dataset["sequence"].to_numpy(),
    labels=dataset["label"].to_numpy(),
    splits=dataset["split"].to_numpy(),
    groups=dataset["split_group"].to_numpy(),
    model_name=DEFAULT_ESM_MODEL,
)
print("Embedding shape:", embeddings.shape)
print("Saved:", embedding_path)

### 2. Fit Logistic Regression and evaluate held-out data

In [ ]:
status = "synthetic_ui_demo_only" if is_demo else "real_data_demonstration"
artifact, predictions = train_esm_logistic_regression(
    embeddings,
    dataset["label"].to_numpy(),
    dataset["split"].to_numpy(),
    dataset["split_group"].to_numpy(),
    embedding_model_name=DEFAULT_ESM_MODEL,
    scientific_status=status,
    dataset_notes=str(dataset_path),
)
pd.DataFrame(
    [artifact["metadata"]["validation_metrics"], artifact["metadata"]["test_metrics"]],
    index=["validation", "held-out test"],
)[["n", "roc_auc", "average_precision", "balanced_accuracy", "mcc", "f1", "sensitivity_recall", "specificity", "brier_score"]].round(3)

### 3. Save the ESM-2 classifier and compare with the baseline

In [ ]:
model_path = PROJECT_ROOT / ("models/notebook_demo_esm2_logreg.joblib" if is_demo else "models/esm2_logreg.joblib")
prediction_path = PROJECT_ROOT / ("outputs/notebook_demo_esm2_predictions.csv" if is_demo else "outputs/esm2_predictions.csv")
save_artifact(artifact, model_path)
predictions.insert(0, "sequence", dataset["sequence"].to_numpy())
predictions.to_csv(prediction_path, index=False)
print("Saved model:", model_path)

baseline_path = PROJECT_ROOT / ("models/notebook_demo_baseline_rf.joblib" if is_demo else "models/baseline_rf.joblib")
if baseline_path.exists():
    baseline = load_artifact(baseline_path)
    comparison = pd.DataFrame(
        [baseline["metadata"]["test_metrics"], artifact["metadata"]["test_metrics"]],
        index=["Feature Random Forest", "ESM-2 + Logistic Regression"],
    )[["roc_auc", "average_precision", "balanced_accuracy", "mcc", "f1"]]
    display(comparison.round(3))
    axis = comparison[["average_precision", "mcc", "balanced_accuracy"]].plot.bar(
        figsize=(9, 5), color=["#2F6BFF", "#D59B2D", "#667085"], rot=0
    )
    axis.set_ylim(0, 1)
    axis.set_ylabel("Held-out test metric")
    axis.set_title("Models evaluated on identical held-out rows")
    axis.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    comparison_path = PROJECT_ROOT / "outputs/figures/03_model_comparison.png"
    plt.savefig(comparison_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", comparison_path)
else:
    print("Baseline artifact not found; run notebook 02 before model comparison.")

## Checks

In [ ]:
assert embeddings.ndim == 2
assert embeddings.shape[0] == len(dataset)
assert predictions["score"].between(0, 1).all()
assert set(predictions["split"]) == {"train", "validation", "test"}
assert artifact["embedding_model_name"] == DEFAULT_ESM_MODEL
assert artifact["metadata"]["test_metrics"]["n"] == int((dataset["split"] == "test").sum())
print("Embedding alignment, score range, split, model-name, and held-out-count checks passed.")
if is_demo:
    print("Do not share these toy metrics. Rerun after creating data/processed/amp_dataset.csv.")

## Next Steps

1. Investigate where the two models disagree.
2. Report the same held-out metrics and split details for both.
3. If ESM-2 performs worse, discuss small-data limitations instead of hiding the result.
4. Use external validation before making any discovery claim.